In [1]:
import pandas as pd
import numpy as np
import torch 


In [2]:
df = pd.read_csv('data/data.csv', index_col=1, parse_dates=True)

df

,Unnamed: 0,prcp,tmax,tmin,vp,srad,target
date,,,,,,,
1980-01-01,0,7.719036,2.635163,-7.404501,356.909575,108.665846,537.0
1980-01-02,1,3.660781,2.484138,-7.616933,350.118484,123.689874,570.0
1980-01-03,2,4.320509,0.016926,-5.137492,418.422967,74.293477,590.0
1980-01-04,3,13.075693,-1.130720,-15.443935,194.276939,143.881449,524.0
1980-01-05,4,15.739118,0.715120,-10.464246,281.074570,125.166372,594.0
...,...,...,...,...,...,...,...
2014-12-27,12770,14.423401,-3.103501,-6.586893,375.666898,71.825399,-999.0
2014-12-28,12771,10.665780,-2.270882,-8.514715,328.833729,112.132569,-999.0
2014-12-29,12772,0.000000,-8.595164,-19.983440,131.992812,202.927692,-999.0


In [3]:
df[df==-999.0] = np.nan
df_training = df[df.index < '2005-01-01']
df_validation = df[(df.index >= '2005-01-01') & (df.index < '2010-01-01')]
df_test = df[df.index >= '2010-01-01']


In [5]:
from src.data.preprocessing import TimeSeriesDataPreprocessor

In [14]:
                #  historic_input_sequence_length: int, 
                #  forecasting_horizon: int, 
                #  historic_cols: List[str], 
                #  future_cols: List[str], 
                #  target_col: str, 
                #  mean_training_set: pd.Series, 
                #  std_training_set: pd.Series, 
                #  max_nan_pct: float = 0.0, 
                #  device: str = 'cpu', 
                #  dtype: torch.dtype = torch.float32

data_meta_data_obj = TimeSeriesDataPreprocessor(
    historic_input_sequence_length=30,
    forecasting_horizon=7,
    historic_cols=['prcp', 'tmax', 'vp', 'srad'],
    future_cols=['prcp', 'tmax', 'tmin', 'vp', 'srad'],
    target_col='target',
    mean_training_set=df_training.mean(),
    std_training_set=df_training.std(),
    max_nan_pct=0,
    device='cpu',
    dtype=torch.float32
)

ds = data_meta_data_obj.make_torch_dataloader([df])

In [24]:
ds[0][0].shape, ds[0][1].shape, ds[0][2].shape

(torch.Size([30, 8]), torch.Size([7, 9]), torch.Size([7, 1]))

In [15]:
data_meta_data_obj.mean_training_set

Unnamed: 0    4562.000000
prcp             3.611909
tmax            10.508491
tmin            -1.634795
vp             587.441343
srad           321.747159
target        2687.902027
dtype: float64

In [31]:
# make a mock df with only 0 and 1 values
df_static = pd.DataFrame({
    'static_col_1': [0, 1, 0, 1, 0, 1],
    'static_col_2': [1, 0, 1, 0, 1, 0.5]
})
static_cols = ['static_col_1', 'static_col_2']

one_hot_cols = [col for col in static_cols if (1.0 in df_static[col].unique() and 0.0 in df_static[col].unique() and df_static[col].nunique() == 2)]

one_hot_cols

['static_col_1']

In [11]:
(df - data_meta_data_obj.mean_training_set) / data_meta_data_obj.std_training_set

,Unnamed: 0,prcp,tmax,tmin,vp,srad,target
date,,,,,,,
1980-01-01,-1.731766,0.725191,-0.783591,-0.838600,-0.874679,-1.327163,-0.618247
1980-01-02,-1.731386,0.008629,-0.798622,-0.869476,-0.900446,-1.233587,-0.608761
1980-01-03,-1.731007,0.125117,-1.044170,-0.509101,-0.641286,-1.541249,-0.603013
1980-01-04,-1.730627,1.671009,-1.158390,-2.007095,-1.491737,-1.107825,-0.621983
1980-01-05,-1.730248,2.141287,-0.974683,-1.283320,-1.162411,-1.224390,-0.601863
...,...,...,...,...,...,...,...
2014-12-27,3.115812,1.908972,-1.354730,-0.719765,-0.803511,-1.556621,NaN
2014-12-28,3.116192,1.245494,-1.271864,-0.999964,-0.981204,-1.305570,NaN
2014-12-29,3.116572,-0.637751,-1.901286,-2.666891,-1.728055,-0.740059,NaN


In [ ]:
# # at the top of your notebook
# %load_ext autoreload
# %autoreload 2

from src.models.LSTM import LSTMHistoric



ImportError: cannot import name 'DataPreprocessor' from 'src.data.preprocessing' (c:\Users\christiaan.wewer\Documents\eigen_projectjes\deep_timeseries_forecaster\src\data\preprocessing.py)

In [ ]:
fc = Forecaster(
    model=LSTMHistoric,
    model_config={
        'LSTM_hidden_size':             32, 
        'LSTM_num_layers':              1, 
        'dropout_rate':                 0.1, 
        },
    name='LSTM_test',
    dfs_training_sets=[df_training],
    dfs_validation_sets=[df_validation],   
    historic_cols=['prcp', 'tmax', 'tmin', 'vp', 'srad'],
    future_cols=['prcp'],
    save_path='models',
    batch_size=1024,
    number_of_epochs=1
)

fc.compute_nummber_of_parameters()
fc.fit()

/home/Christiaan/.conda/envs/thesis/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


18631

In [ ]:
fc_min = Forecaster(
    model=minLSTMHistoric,
    model_config={
        'minLSTM_hidden_size':             32, 
        'minLSTM_num_layers':              2, 
        'dropout_rate':                 0.1, 
        },
    name='minLSTM_test',
    dfs_training_sets=[df_training],
    dfs_validation_sets=[df_validation],   
    historic_cols=['prcp', 'tmax', 'tmin', 'vp', 'srad'],
    future_cols=['prcp'],
    save_path='models',
    batch_size=1024,
)

fc_min.compute_nummber_of_parameters()
fc_min.fit()

x_decoded.shape
torch.Size([1024, 1, 7])


/home/Christiaan/.conda/envs/thesis/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([1024, 7, 1])) that is different to the input size (torch.Size([1024, 1, 7])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


x_decoded.shape
torch.Size([1024, 1, 7])
x_decoded.shape
torch.Size([1024, 1, 7])


KeyboardInterrupt: 

In [ ]:
# fc.fit()

Model saved to models/test/model_test.pt
Epoch 1/100 - Training Loss: 0.7168000936508179 - Validation Criterion: 0.3424745500087738 - Best Validation Criterion: 0.3424745500087738


KeyboardInterrupt: 

In [ ]:
from ForecastingModel.models.minLSTM import minLSTM